In [ ]:
def _topic_to_path(
    base_path: str,
    topic: str,
) -> str:
    topic_path = topic.replace(".", "/")

    return (
        f"{base_path.rstrip('/')}/"
        f"{topic_path}"
    )

In [ ]:
required_columns = {
    "event_timestamp",
    "event_date",
    "kafka_topic",
    "kafka_partition",
    "kafka_offset",
}
missing_columns = (
    required_columns
    - set(test_df.columns)
)

if missing_columns:
    raise ValueError(
        "MinIO sink received an invalid DataFrame. "
        f"Missing columns: {sorted(missing_columns)}"
    )
configure_minio_storage(spark)

base_path = "s3a://commerceflow-lakehouse/_smoke_tests"

topics = [
    row["kafka_topic"]
    for row in (
        test_df
        .select("kafka_topic")
        .distinct()
        .collect()
    )
]

if len(topics) != 1:
    raise ValueError(
        "The test DataFrame must contain exactly one Kafka topic. "
        f"Found topics: {topics}"
    )

topic = topics[0]

output_path = _topic_to_path(
    base_path=base_path,
    topic=topic,
)

print(f"Kafka topic: {topic}")
print(f"Output path: {output_path}")

(
    test_df
    .repartition("event_date")
    .write
    .mode("overwrite")
    .partitionBy("event_date")
    .parquet(output_path)
)

print("Test data written successfully.")

